# Laboratorio 5 — Navegación Basada en Comportamientos

Jesús Manuel Aragón Buitrago

Juan Carlos González Ibarra

Ángel Rivera Amórtegui

Duvan Stiven Tique Osorio

Universidad Nacional de Colombia — Facultad de Ingeniería

Laboratorio de Sistemas Inteligentes en Robótica (Labsir)

Curso: Fundamentos de Robótica Móvil

## Introducción

En este laboratorio el robot EV3 navega de forma autónoma, sin un mapa previo del entorno, reaccionando en tiempo real a la información de sus sensores. Se realizaron dos pruebas: evadir obstáculos con un algoritmo Bug y resolver un laberinto siguiendo una pared.

## Investigación preliminar

**Navegación planeada vs. reactiva:** la navegación planeada calcula toda la ruta de antemano a partir de un mapa del entorno; es más precisa, pero falla si el entorno cambia. La navegación reactiva no usa mapa: el robot decide su movimiento según lo que detectan sus sensores en cada instante; es menos ordenada, pero responde mejor a imprevistos.

**Rodney Brooks:** investigador del MIT, propuso en los años 80 la arquitectura de subsunción, en la que el control del robot se divide en capas simples de comportamiento (evitar obstáculos, explorar, avanzar) que se ejecutan en paralelo sin necesidad de un modelo interno del mundo ni de planeación simbólica. Su idea central, resumida en el artículo "Intelligence Without Representation" (1991), es que "el mundo es su propio mejor modelo": el robot puede actuar de forma inteligente reaccionando directamente al entorno, sin construir ni mantener un mapa. Aplicó estos principios en robots como Genghis y Allen, capaces de caminar y evadir obstáculos con muy poco cómputo, y más adelante fundó iRobot (creadora del Roomba) y Rethink Robotics.

**Mark Tilden:** físico e ingeniero conocido por crear la robótica BEAM (Biology, Electronics, Aesthetics, Mechanics), un enfoque en el que los robots se controlan con circuitos analógicos simples —redes de "neuronas" nerviosas hechas con transistores— en lugar de un microprocesador o código. Sus robots, como los solar rollers y caminantes solares alimentados por celdas fotovoltaicas, imitan reflejos biológicos básicos (moverse hacia la luz, retroceder al chocar) sin ningún tipo de programación. Este enfoque influyó en el diseño de robots comerciales de bajo costo, como el Robosapien, y demuestra que se puede lograr comportamiento adaptativo sin inteligencia artificial tradicional.

**Algoritmos de planeación de rutas entre obstáculos (3 ejemplos):**
- Campos de potencial: la meta atrae al robot y los obstáculos lo repelen.
- Grafos de visibilidad: se traza el camino más corto conectando las esquinas de los obstáculos.
- Búsqueda en grilla (A*): el espacio se divide en celdas y se busca el camino más corto entre ellas.

**Algoritmos Bug (para evadir obstáculos):**
- **Bug 0:** avanza en línea recta hacia la meta; si se topa con un obstáculo, lo bordea hasta poder retomar la línea recta.
- **Bug 1:** recorre todo el contorno del obstáculo, memoriza el punto más cercano a la meta, y continúa desde ahí.
- **Bug 2:** sigue una línea imaginaria hacia la meta; si choca, bordea el obstáculo hasta volver a cruzar esa línea más cerca de la meta.

**Algoritmo de laberintos:** consiste en mantener contacto con una misma pared (derecha o izquierda) durante todo el recorrido. Si el laberinto no tiene zonas aisladas, este método siempre lleva a la salida.


## Misión 1: Evadir obstáculos (algoritmo Bug)

**Objetivo:** ir de forma autónoma de P1 a P2 (marcados con cinta en el piso) evadiendo al menos dos obstáculos.

**Cómo se resolvió:** el robot sigue la cinta con el sensor de color. Al chocar con un obstáculo (detectado por el sensor de contacto), retrocede, gira y lo bordea usando el sensor ultrasónico para mantener una distancia aproximadamente constante. Cuando vuelve a detectar la cinta, retoma el seguimiento de línea. Se detiene al detectar el color de meta en P2.

**Pseudocódigo:**

```
seguir la línea
si el sensor de contacto se activa:
    retroceder, girar, bordear el obstáculo usando el ultrasónico
    al reencontrar la línea, retomar el seguimiento de línea
si se detecta el color de la meta:
    detener el robot
```



**Programa:**


In [ ]:
from time import sleep, time
from ev3dev2.motor import LargeMotor, OUTPUT_B, OUTPUT_C, SpeedPercent
from ev3dev2.sensor.lego import TouchSensor, ColorSensor, UltrasonicSensor

# --- Hardware ---
MotIzq = LargeMotor(OUTPUT_B)
MotDer = LargeMotor(OUTPUT_C)
toque  = TouchSensor('in4')
color  = ColorSensor('in3')
usonic = UltrasonicSensor('in1')

color.mode  = 'COL-COLOR'
usonic.mode = 'US-DIST-CM'

# --- Parámetros ---
VEL_BASE     = 20
DIST_CERCA   = 15   
DIST_LEJOS   = 20    
TIEMPO_MIN   = 1.5   

# --- Estados ---
SIGUIENDO_LINEA = 0
RODEANDO        = 1
estado          = SIGUIENDO_LINEA
tiempo_rodeo    = 0


def en_linea():
    return color.color == 4


def seguirLinea():
    if color.color == 4:
        MotIzq.on(SpeedPercent(VEL_BASE))
        MotDer.on(SpeedPercent(VEL_BASE))
    else:
        MotIzq.on(SpeedPercent(0))
        MotDer.on(SpeedPercent(VEL_BASE))


def rodearObstaculo():
    dist = usonic.distance_centimeters
    if dist > DIST_LEJOS:
        MotIzq.on(SpeedPercent(20))   
        MotDer.on(SpeedPercent(5))
    elif dist < DIST_CERCA:
        MotIzq.on(SpeedPercent(5))  
        MotDer.on(SpeedPercent(20))
    else:
        MotIzq.on(SpeedPercent(15)) 
        MotDer.on(SpeedPercent(15))


def girar90Izquierda():
    MotIzq.reset()
    MotDer.reset()
    MotIzq.on_for_degrees(SpeedPercent(-25), 180, brake=True, block=False)
    MotDer.on_for_degrees(SpeedPercent(25),  180, brake=True, block=False)
    while MotIzq.is_running or MotDer.is_running:
        sleep(0.02)


# --- Loop principal ---
try:
    while True:
        if estado == SIGUIENDO_LINEA:
            if color.color == 5:
                MotIzq.off()
                MotDer.off()
                break
            elif toque.is_pressed:
                MotIzq.on(SpeedPercent(-20),1)   
                MotDer.on(SpeedPercent(-20),1)
                girar90Izquierda()                  
                tiempo_rodeo = time()
                estado = RODEANDO
            else:
                seguirLinea()

        elif estado == RODEANDO:
            rodearObstaculo()
            if (time() - tiempo_rodeo) > TIEMPO_MIN and en_linea():
                MotIzq.off()
                MotDer.off()
                estado = SIGUIENDO_LINEA

        sleep(0.05)
finally:
    MotIzq.off()
    MotDer.off()


**Video de ejecución:**

<a href="https://youtu.be/jJsZnpXOxZY" target="_blank">Misión 1 — Bug</a>

## Misión 2: Superar el laberinto (algoritmo Maze)

**Objetivo:** recorrer de forma autónoma un laberinto desde la entrada (P1) hasta la salida (P2), pasando por al menos un pasillo recto, un callejón sin salida y una intersección.

**Cómo se resolvió:** el robot avanza en línea recta hasta que el sensor infrarrojo detecta una pared al frente. En ese punto revisa con el sensor ultrasónico si hay pared a la derecha: si la hay, gira 90° a la izquierda; si no la hay, gira 90° a la derecha. Así mantiene siempre contacto con la pared derecha.

**Nota:** falta agregar una condición para que el robot detecte la salida y se detenga; en la versión actual el recorrido no tiene fin.

**Pseudocódigo:**

```
avanzar hasta detectar pared al frente
si hay pared a la derecha:
    girar a la izquierda
si no:
    girar a la derecha
[PENDIENTE: detener el robot al llegar a la salida]
```

**Programa:**


In [ ]:
from time import sleep
from ev3dev2.motor import LargeMotor, OUTPUT_B, OUTPUT_C, SpeedPercent
from ev3dev2.sensor.lego import InfraredSensor, UltrasonicSensor

# --- Hardware ---
MotIzq = LargeMotor(OUTPUT_B)
MotDer = LargeMotor(OUTPUT_C)
infra  = InfraredSensor('in4')
usonic = UltrasonicSensor('in3')

infra.mode  = 'IR-PROX'
usonic.mode = 'US-DIST-CM'

# --- Parámetros ---
VEL_BASE    = 20
GRADOS_GIRO = 180
UMBRAL_IR   = 6    
UMBRAL_US   = 20   # cm 


def hay_pared_frente():
    return infra.proximity < UMBRAL_IR


def hay_pared_derecha():
    return usonic.distance_centimeters < UMBRAL_US


def avanzar():
    while not hay_pared_frente():
        MotIzq.on(SpeedPercent(VEL_BASE))
        MotDer.on(SpeedPercent(VEL_BASE))
        sleep(0.05)
    MotIzq.off()
    MotDer.off()


def girar90Derecha():
    MotIzq.reset()
    MotDer.reset()
    MotIzq.on_for_degrees(SpeedPercent(25),  GRADOS_GIRO, brake=True, block=False)
    MotDer.on_for_degrees(SpeedPercent(-25), GRADOS_GIRO, brake=True, block=False)
    while MotIzq.is_running or MotDer.is_running:
        sleep(0.02)


def girar90Izquierda():
    MotIzq.reset()
    MotDer.reset()
    MotIzq.on_for_degrees(SpeedPercent(-25), GRADOS_GIRO, brake=True, block=False)
    MotDer.on_for_degrees(SpeedPercent(25),  GRADOS_GIRO, brake=True, block=False)
    while MotIzq.is_running or MotDer.is_running:
        sleep(0.02)


try:
    while True:
        avanzar()
        if hay_pared_derecha():
            girar90Izquierda()
        else:
            girar90Derecha()
finally:
    MotIzq.off()
    MotDer.off()


**Video de ejecución:**

<a href="https://youtube.com/shorts/uhSBLi5XLyg?feature=share" target="_blank">Misión 2 — Laberinto</a>

## Videos de ejecución

- <a href="https://youtu.be/jJsZnpXOxZY" target="_blank">Misión 1 — Bug</a>
- <a href="https://youtube.com/shorts/uhSBLi5XLyg?feature=share" target="_blank">Misión 2 — Laberinto</a>

## Experimentación y dificultades

**1. Retroceso incompleto antes de girar en `Bug.py`.** El código busca que el robot retroceda 1 segundo antes de girar hacia el bordeo, mediante `MotIzq.on(SpeedPercent(-20), 1)`. Sin embargo, ese método solo enciende el motor y el programa continúa de inmediato a la siguiente instrucción, sin esperar ningún tiempo; como no hay un `sleep()` entre el retroceso y el giro, el robot apenas retrocede una fracción de segundo antes de empezar a girar. La solución hubiera sido agregar un `sleep(1)` (o usar `on_for_seconds()`) entre esas dos acciones, pero en la práctica no fue necesario: el algoritmo de bordeo funcionó correctamente de todas formas, ya que el giro de 90° alcanzó a separar al robot lo suficiente del obstáculo.

**2. El robot no se detiene al llegar a la salida del laberinto.** El bucle principal de `laberinto.py` no tiene una condición de término, por lo que el robot debe detenerse manualmente al llegar a P2. La solución planteada fue agregar un sensor de color con una marca distintiva en la salida, pero no había puertos físicos disponibles en el robot para conectarlo. Una alternativa que no requiere un sensor adicional es usar el encoder de los motores (ya usado para los giros de 90°) para medir la distancia recorrida en línea recta: si el robot avanza más lejos de lo que mide cualquier pasillo interno del laberinto sin encontrar pared al frente, es señal de que salió a espacio abierto. Esta alternativa no se llegó a implementar antes de la entrega.



**Conclusión:** las dos dificultades encontradas fueron de tipo software y no de diseño del comportamiento: en ambos casos la lógica de decisión (bordear el obstáculo, seguir la pared derecha) era correcta, pero faltaron detalles de implementación (temporización y condición de término) que no impidieron completar las misiones. Esto muestra que, en navegación reactiva, comportamientos simples pueden tolerar imprecisiones de bajo nivel sin dejar de cumplir el objetivo general.
